# [실습 03] 프로파일 + 함수 호출 미니 에이전트

> **연계**: 제2부 03장(4대 모듈) · **환경**: Google Colab · **모델**: 오픈웨이트 `Qwen/Qwen2.5-1.5B-Instruct`

**학습 목표**
- **프로파일(03-1)** 로 에이전트의 역할·규칙을 정한다.
- **도구 사용(03-4)** 을 함수 호출 형태로 구현하고, LLM이 도구를 고르게 한다.

In [ ]:
!pip install -q transformers accelerate torch

## 1. 도구(함수) 정의 — 에이전트의 '손과 발'

In [ ]:
def get_weather(city):
    fake = {"서울": "맑음 28도", "부산": "흐림 25도"}
    return fake.get(city, "정보 없음")

def calculator(expr):
    return str(eval(expr, {"__builtins__": {}}, {}))

TOOLS = {"get_weather": get_weather, "calculator": calculator}
print(get_weather("서울"), calculator("12*3"))

## 2. 프로파일(시스템 프롬프트)로 역할·형식 지정

LLM이 도구를 정확한 **형식**(JSON)으로 부르도록 프로파일에 규칙을 심습니다. (01-2 지시 따르기)

In [ ]:
import torch, json, re
from transformers import pipeline

gen = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

PROFILE = '''너는 도구를 쓰는 비서다. 도구가 필요하면 반드시 아래 JSON만 출력하라.
{"tool": "get_weather|calculator", "arg": "..."}
도구: get_weather(city), calculator(expr)'''

## 3. LLM이 도구를 고르고, 애플리케이션이 실행 (판단·실행 분리)

In [ ]:
def run(user):
    msg = [{"role": "system", "content": PROFILE},
           {"role": "user", "content": user}]
    out = gen(msg, max_new_tokens=80, do_sample=False)[0]["generated_text"][-1]["content"]
    m = re.search(r'\{.*\}', out, re.S)
    if not m:
        return "(도구 미사용) " + out
    call = json.loads(m.group())
    result = TOOLS[call["tool"]](call["arg"])
    return f"도구 {call['tool']}({call['arg']}) → {result}"

print(run("서울 날씨 알려줘"))
print(run("25 곱하기 4는?"))

## 4. 정리
- **프로파일**로 역할·출력 형식을 고정하고, **함수 호출**로 도구를 사용했다.
- LLM은 '무엇을 부를지'만 판단하고 실행은 애플리케이션이 담당한다(판단·실행 분리).
- **더 해보기**: 새 도구 `translate(text)`를 추가해 프로파일에 등록해 보세요.